# Experiment 002 on Google Colab Pro+ (확정 기록 10: 실행 환경 서버 -> Colab)
Run the cells top to bottom. Every cell is idempotent: after a disconnect, choose a GPU runtime and run all cells again.

**Training resumes from the last checkpoint** (user decision; an earlier "no resume" instruction of 2026-11-04 was a user mistake and was corrected the same day). This needs the resume-patched `train.py` (cell 5 applies `colab/train_resume.patch` on the pinned commit): the patch restores the encoders and the optimizer states and reads `training.epochs` as the TOTAL target epoch, so a run resumed at epoch 5 trains epochs 6..15 and the total stays at 27855 optimizer steps. The unpatched `train.py` would train `epochs` MORE epochs; `trainrun.py` therefore refuses to continue a checkpoint with an unpatched `train.py`.
Before every train attempt `colab_tools.py prepare` runs: loadable `model_latest.pth` -> resume; broken `model_latest.pth` -> replaced by the newest loadable `model_<e>.pth` -> resume; no loadable checkpoint -> checkpoints are MOVED (not deleted) to `WORK_ROOT/train_runs/quarantine/<run>__<time>/` and the run starts from epoch 0.
Limits of resume (추정, not verified on cloth / torch 2.0.1 / Colab): the data order and RNG state are not restored, so a resumed run need not be bit-identical to an uninterrupted one; the epoch in progress at the disconnect is recomputed (at most one epoch with save_every_x_epoch=1).

Operational notes (short)
- Persistent state lives in Drive: `WORK_ROOT` (default `MyDrive/exp002_work`) holds `train_runs/` (checkpoints, DONE, quarantine), `exp002_cache/`, `results/` (incl. `attempts.json`), `manifests/`, `logs/`. The local disk (`/content`) is lost when the session ends (venv, repo clone, copied data are rebuilt each session).
- Training runs in the background (own session), so closing the tab does not stop it **only if** the runtime stays alive (Colab Pro+ background execution). The idle rule (about 90 min without interaction) and the maximum session length (about 24 h) still apply; these limits are from memory of Colab's documentation (추정), not checked today. After a disconnect re-run the last cell.
- Do NOT change seeds, batch_size 24, num_hist 1, epochs 15 or any pre-registered threshold here. GPU training needs human approval (CLAUDE.md).

In [ ]:
# 1. GPU / Python check
import sys, subprocess
def sh(cmd, cwd=None):
    # run a shell command and stream its output into the cell
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    return p.wait()
sh('nvidia-smi')
print(sys.version)
print(subprocess.run("df -h /content | tail -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 2. Mount Drive and choose the persistent work root
import os
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT = os.environ.get('EXP002_WORK_ROOT', '/content/drive/MyDrive/exp002_work')
os.makedirs(WORK_ROOT, exist_ok=True)
# Data folder (must contain ind_train, a060_train, ..., a248_test). The user added a shortcut 'dino-wm-videos' (id 17M0MZGSDPzu7CCT3ez3ABMyZGjavTfx3, owned by another
# account) to MyDrive; per the user's Drive-metadata check the config folder id (data_impl.drive.root_folder_id) is '[data]/[cloth-sim] cloth_move' inside it.
# The names contain brackets and a space: this path is only passed through os.environ / pathlib, never through a shell command line.
# Expected mounted path (UNVERIFIED; differs if the shortcut was renamed). Fallback if it does not exist: the code searches below
# /content/drive/MyDrive/dino-wm-videos (depth <= 3) for a folder that has all split directories; then /content/drive/.shortcut-targets-by-id/<root_folder_id>.
DATA_SRC = os.environ.get('EXP002_DRIVE_MOUNT_SRC', '/content/drive/MyDrive/dino-wm-videos/[data]/[cloth-sim] cloth_move')
print('WORK_ROOT =', WORK_ROOT, '| DATA_SRC =', DATA_SRC, '| exists:', os.path.isdir(DATA_SRC))

In [ ]:
# 3. Repo clone / pull (branch exp002-single-size-sweep).
# REPO_URL equals `git remote get-url origin` of the lab repo. On 2026-11-04 an unauthenticated `git ls-remote` on this URL (and on the thread0 URL in cell 5)
# listed the branches from the authoring PC, so both appear PUBLIC and need no token; the repository settings were not inspected (public/private status is
# a conclusion from that one test, not confirmed). If a clone ever asks for credentials (repo made private), put a token in Colab Secrets as GITHUB_TOKEN.
# NOTE: this clones the REMOTE branch: commit and push the colab/ files first (remote branch tip on 2026-11-04 was 10e2833, without them).
import os, subprocess
REPO_URL = 'https://github.com/jongmin-s-multicore/probing-experiment.git'
BRANCH = 'exp002-single-size-sweep'
REPO = '/content/mylab'
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', 'https://%s@' % tok)
except Exception:
    pass
if not os.path.isdir(REPO + '/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH], check=True)
EXP = REPO + '/experiments/002_single_size_sweep'
assert os.path.isfile(EXP + '/colab/colab_tools.py'), 'colab/ files are missing on this branch: commit and push them first'
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# 4. Environment: pip (no conda on Colab; documented exception to CLAUDE.md, see README "Colab").
# Default: Python 3.9 venv via uv with the environment.yml pins (torch 2.0.1+cu117, numpy 1.26.4 ...), rebuilt every new session (a few minutes).
# Fallback USE_SYSTEM_PYTHON=1: Colab's own Python/torch (versions differ from environment.yml; results are then NOT comparable, record it).
import os, subprocess, sys
USE_SYSTEM_PYTHON = False
VENV = '/content/venv'
if USE_SYSTEM_PYTHON:
    PY = sys.executable
else:
    PY = VENV + '/bin/python'
    if not os.path.exists(VENV + '/.exp002_ok'):
        assert sh('pip install -q uv') == 0
        assert sh('uv venv --python 3.9 %s' % VENV) == 0
        assert sh('uv pip install --python %s --index-strategy unsafe-best-match -r %s/colab/requirements_colab.txt' % (PY, EXP)) == 0
        open(VENV + '/.exp002_ok', 'w').write('ok\n')
CHK = "import sys,torch,numpy,cv2,decord,hydra;print(sys.version.split()[0],'torch',torch.__version__,'cuda',torch.version.cuda,torch.cuda.is_available(),'numpy',numpy.__version__)"
sh('%s -c "%s"' % (PY, CHK))

In [ ]:
# 5. ACWM-Phys checkout at the pinned commit (== config.yaml code.commit) + resume patch on train.py only.
# URL == config.yaml code.repo_url (public at the 2026-11-04 unauthenticated ls-remote test; tip of hanbin5/acwm-phys was 5a640bed there).
# The patch (user-approved 2026-11-04): restores encoders + optimizer states from model_latest.pth and reads training.epochs as the TOTAL target epoch.
# A fresh run (epoch 0) runs epochs 1..epochs exactly like the original.
import hashlib, os, subprocess
COMMIT = '5a640bed66002eca7bdb31d10b2bde59ad944551'
SRC = '/content/dinowm/acwm_phys_src'
if not os.path.isdir(SRC + '/.git'):
    os.makedirs(os.path.dirname(SRC), exist_ok=True)
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/jongmin-s-multicore/thread0.git', SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'checkout', '--detach', COMMIT], check=True)
head = subprocess.run(['git', '-C', SRC, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, head
PATCH = EXP + '/colab/train_resume.patch'
if subprocess.run(['git', '-C', SRC, 'apply', '--reverse', '--check', PATCH], capture_output=True).returncode == 0:
    print('resume patch already applied')
else:
    subprocess.run(['git', '-C', SRC, 'apply', PATCH], check=True)
    print('resume patch applied')
dirty = subprocess.run(['git', '-C', SRC, 'status', '--porcelain', '--untracked-files=no'], capture_output=True, text=True).stdout.split('\n')
assert [d for d in dirty if d.strip()] == [' M train.py'], dirty      # only train.py may differ from the pinned commit
sha = hashlib.sha256(open(PATCH, 'rb').read()).hexdigest()
open(WORK_ROOT + '/acwm_resume_patch.sha256', 'w').write(sha + '\n')
print('commit', head, '| patch sha256', sha)

In [ ]:
# 6. Environment variables (resolve_paths reads these) + Drive-backed results/manifests/logs
import os, subprocess
os.environ.update({
    'DINO_WORK': '/content/dinowm',
    'DATASET_DIR': '/content/dinowm/data',                 # local copy (fast reads)  -> data_root = DATASET_DIR/cloth_sim
    'DINO_TRAIN': WORK_ROOT + '/train_runs',               # Drive: checkpoints, DONE markers
    'EXP002_CACHE': WORK_ROOT + '/exp002_cache',           # Drive
    'ACWM_PHYS_SRC': SRC,
    'WANDB_MODE': 'disabled',
    'EXP002_WORK_ROOT': WORK_ROOT,                         # archive lookup/creation: WORK_ROOT/cloth_sim_data.tar (or .zip)
    'EXP002_DATA_MODE': 'mount',                           # download stage copies from the mounted Drive, no rclone
    'PATH': os.path.dirname(PY) + os.pathsep + os.environ['PATH'],
})
if DATA_SRC:
    os.environ['EXP002_DRIVE_MOUNT_SRC'] = DATA_SRC
if not USE_SYSTEM_PYTHON:
    os.environ['VIRTUAL_ENV'] = VENV
sh('%s %s/colab/colab_tools.py setup-dirs --work-root %s' % (PY, EXP, WORK_ROOT))

In [ ]:
# 7. Data -> local /content. Thousands of small mp4 reads through the Drive mount are slow, so (estimate, not measured) one big archive file is faster:
#    a) WORK_ROOT/cloth_sim_data.tar (uncompressed tar; mp4 is already compressed) or cloth_sim_data.zip exists -> ONE sequential copy to /content, extract locally,
#       then the same verification as before (per split: mp4 count == metadata length == expected, total bytes recorded). Layouts cloth_sim/<split>/..., <split>/...
#       or <any>/<split>/... are accepted; absolute paths, '..' and links are refused.
#    b) no archive -> parallel copy (EXP002_COPY_WORKERS, default 16) from the mounted folder (DATA_SRC / fallback search), verification, and then the archive
#       is written once to WORK_ROOT (tmp name + rename; set EXP002_MAKE_ARCHIVE=0 to skip). The next session uses only the archive.
#    Own archive instead: on your PC  tar -cf cloth_sim_data.tar cloth_sim   (folder cloth_sim/ containing the *_train / *_test split folders)  or a zip of the
#    same folder, upload it to MyDrive/exp002_work/. A damaged archive is never modified: delete/rename it to fall back to (b).
#    Complete splits are skipped; an interrupted run is safe to re-run (*.part temp files, short files are copied again).
import subprocess
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)
rc = sh('%s run.py download' % PY, cwd=EXP)
assert rc == 0, 'download/copy failed (rc=%d); read the lines above' % rc

In [ ]:
# 8. check / manifest / (optional) smoke. Smoke is a GPU run: only with human approval; skipped when results/smoke.json exists.
RUN_SMOKE = False
sh('%s run.py check' % PY, cwd=EXP)
assert sh('%s run.py manifest' % PY, cwd=EXP) == 0
import os
if RUN_SMOKE and not os.path.exists(EXP + '/results/smoke.json'):
    sh('%s run.py smoke' % PY, cwd=EXP)

In [ ]:
# 9. Full run (= overnight.sh equivalent): train x (2 conditions x 3 seeds) -> cache -> q1 -> q2 -> q3 -> aggregate.
# Re-running this cell after a disconnect: DONE stages are skipped; a run WITHOUT DONE resumes from its last valid checkpoint (patched train.py), or starts from epoch 0 if there is none.
# GPU training: start only after human approval.
import os, subprocess, time
CONDITIONS = 'multi_60_ind_160 single_ind'                 # config conditions.run_order without the optional small condition
env = dict(os.environ, CONDITIONS=CONDITIONS, RETRY_SLEEP='60')
running = subprocess.run('pgrep -f "[b]ash overnight.sh"', shell=True, capture_output=True).returncode == 0
if not running:
    # nothing is training now: keep valid checkpoints (resume), repair a half-written model_latest.pth, quarantine (move, never delete) unusable ones.
    # overnight.sh repeats this before each train attempt (also retries), so this call is a visible duplicate, not the only guard.
    sh('%s %s/colab/colab_tools.py prepare --train-root %s/train_runs' % (PY, EXP, WORK_ROOT))
LOG = EXP + '/logs/overnight.log'
if running:
    print('overnight.sh is already running in this session; attaching to its log')
else:
    p = subprocess.Popen('bash overnight.sh >> %s 2>&1' % LOG, shell=True, cwd=EXP, env=env, start_new_session=True)
    print('started overnight.sh pid', p.pid)
try:
    while subprocess.run('pgrep -f "[b]ash overnight.sh"', shell=True, capture_output=True).returncode == 0:
        print(subprocess.run(['tail', '-n', '3', LOG], capture_output=True, text=True).stdout)
        subprocess.run([PY, EXP + '/colab/colab_tools.py', 'prune', '--train-root', WORK_ROOT + '/train_runs'])
        time.sleep(300)          # keeps the cell busy; interrupting the cell does not stop the background job
except KeyboardInterrupt:
    print('cell interrupted; the background job keeps running')
print(subprocess.run(['tail', '-n', '20', LOG], capture_output=True, text=True).stdout)

In [ ]:
# 10. After everything is finished (or before you leave): flush Drive writes.
from google.colab import drive
drive.flush_and_unmount()